In [2]:
import torch
import numpy as np
from models.scold.model import LVL
from transformers import RobertaTokenizer
from PIL import Image
from torchvision import transforms

Debug: Starting model.py imports...
Debug: torch imported successfully: 2.5.1+cu118
Debug: numpy imported successfully: 2.1.2
Debug: transformers imported successfully
Debug: All imports completed successfully


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Load model
model = LVL()
model.load_state_dict(torch.load("models/scold/scold.pt", map_location=device))
model.to(device)
model.eval()

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\Andakara\AppData\Local\Temp\ipykernel_192056\2665840536.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe

LVL(
  (image_encoder): ImageEncoder(
    (swin): FeatureListNet(
      (patch_embed): PatchEmbed(
        (proj): Conv2d(3, 128, kernel_size=(4, 4), stride=(4, 4))
        (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      )
      (layers_0): SwinTransformerStage(
        (downsample): Identity()
        (blocks): Sequential(
          (0): SwinTransformerBlock(
            (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
            (attn): WindowAttention(
              (qkv): Linear(in_features=128, out_features=384, bias=True)
              (attn_drop): Dropout(p=0.0, inplace=False)
              (proj): Linear(in_features=128, out_features=128, bias=True)
              (proj_drop): Dropout(p=0.0, inplace=False)
              (softmax): Softmax(dim=-1)
            )
            (drop_path1): Identity()
            (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
            (mlp): Mlp(
              (fc1): Linear(in_features=128, out_fe

In [6]:
# Load tokenizer
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

# Image transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [7]:

def predict(image_path, text):
    image = transform(Image.open(image_path).convert("RGB")).unsqueeze(0).to(device)
    tokens = tokenizer(text, return_tensors="pt", padding=True, truncation=True).to(device)

    with torch.no_grad():
        img_feat, txt_feat = model(image, tokens["input_ids"], tokens["attention_mask"])
        similarity = torch.matmul(img_feat, txt_feat.T).squeeze()

    return similarity.item()

In [13]:
predict("apple.jpg", "An apple leaf with yellow rust")

0.5096084475517273